In [ ]:
import glob, os
import ast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from astropy.io import fits
from scipy.optimize import curve_fit

%matplotlib widget

In [ ]:
# toto tot test
def get_colors(n_steps, cmap_name='Blues', vmin=0, vmax=None):
    if vmax is None:
        vmax = n_steps - 1
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap_name)
    return [cmap(norm(i)) for i in range(n_steps)], norm, cmap


# Play with cos functions

In [ ]:
xx = np.arange(0, 360)
plt.figure()
plt.plot(xx, np.cos(np.radians(xx)), label=('cos'))
plt.plot(xx, np.cos(np.radians(xx))**2, label=('cos^2'))
plt.plot(xx, np.cos(np.radians(xx))**4, label=('cos^4'))
plt.grid()
plt.legend()

xx = np.arange(0, 360)
plt.figure()
plt.plot(xx, np.sin(np.radians(xx)), label=('sin'))
plt.plot(xx, np.sin(np.radians(xx))**2, label=('sin^2'))
plt.plot(xx, np.sin(np.radians(xx))**4, label=('sin^4'))
plt.grid()
plt.legend()

In [ ]:
trans = np.arange(0, 50, 45)
print(trans)
ff = np.size(trans)
colors, norm, cmap = get_colors(ff, cmap_name='viridis', vmin=0, vmax=ff-1)

xx = np.arange(0, 360)
plt.figure()
plt.plot(xx, np.cos(np.radians(xx))**4, label=('cos^4'), color='red')
plt.plot(xx, np.cos(np.radians(xx))**2 * np.sin(np.radians(xx))**2, label=('cos^2*sin^2'), color='orange')

for i in range(ff):
    #plt.plot(xx, np.cos(np.radians(trans[i]+xx))**2 * np.sin(np.radians(xx))**2, color=colors[i])
    plt.plot(xx, np.cos(np.radians(trans[i]+xx))**2 * np.cos(np.radians(xx))**2, color=colors[i])
    #plt.plot(xx, np.cos(np.radians(trans[i]-xx))**2 * np.sin(np.radians(xx-trans[i]))**2, color=colors[i])
plt.grid()
plt.legend()



# Tout premier test

On fait tourner le polariseur et on regarde la moyenne du signal.

In [ ]:

S21 = np.array([-75, -62, -52, -46, -42, -39, -37, -36, -35, -35, -35.5, -36.5, -38, -40.5, -44, -49, -56, -70, -75])
nmeas = np.size(S21)
angle = np.arange(nmeas)*10 +78



In [ ]:
tt = np.arange(70, 260)
plt.figure()
plt.plot(angle, S21, 'o')
#plt.plot(tt, (np.cos(np.radians(tt-180)))**4+np.mean(S21**2), 'r-')
plt.xlabel("Angle (°)")
plt.ylabel("S21 (dB)")
plt.grid(10)
plt.show()

# Premiers scans automatisés

In [ ]:
#path_save = '/home/lmousset/Projets_Recherche/COSMOCal/Tests_IAS_2026/Data/CosmoCal_data/'
path_save = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\CosmoCal_data\\"
hdul = fits.open(path_save + "Scan_20260220_173700.fits")
header = hdul[0].header
header


In [ ]:
theta_min = header['THETA_MI']
theta_max = header['THETA_MA']
step = header['STEP']

theta = np.arange(theta_min, theta_max + step, step)
print(theta.shape)

In [ ]:
nfreq = header['POINTS']

In [ ]:
hdul.info()  # View structure
mag = hdul[0].data  # Access magnitude
phi = hdul[1].data  # Access phase

print(mag.shape)  # Check shape of magnitude data

#mag_lin = 10**(mag/10)
#mag_norm = np.zeros_like(mag_lin)
#for s in range(4):
 #   for f in range(nfreq):
  #      mag_norm[:, s, f] = mag_lin[:, s, f]/np.max(mag_lin[:, s, 10])
#mag_norm.shape

In [ ]:
def cos4(alpha, thetaE=0, amp=1):
    return amp *(np.cos(np.radians(thetaE - alpha)))**4

# Ajustement

#popt, pcov = curve_fit(cos4, theta, mag,)
#theta_fit = np.linspace(theta_min, theta_max, 100)

In [ ]:
freqs = [0, 12, 21, 33, 40, 50]
nf = len(freqs)
tt = np.arange(0, 380)
colors, norm, cmap = get_colors(nf, cmap_name='viridis', vmin=0, vmax=nf-1)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()
#for i, f in enumerate(freqs):
    #plt.plot(theta, 10**(mag[:, trace, f]/10), 'o', color=colors[f])
    #plt.plot(theta, mag_norm[:, trace, f], 'o', color=colors[f])
    #ax[0].plot(theta, 10**(mag[:, 0, f]/10), 'o-', color=colors[i])
    #ax[1].plot(theta, 10**(mag[:, 1, f]/10), 'o-', color=colors[i])

ax[0].plot(theta, 10**(np.mean(mag[:, 0, :], axis=1)/10), 'o', color='b')
ax[1].plot(theta, 10**(np.mean(mag[:, 1, :], axis=1)/10), 'o', color= 'b')

ax[0].plot(tt, cos4(tt, 3)*np.max(10**(np.mean(mag[:, 0, :], axis=1)/10)), 'orange')
ax[1].plot(tt, cos4(tt, 3)*np.max(10**(np.mean(mag[:, 1, :], axis=1)/10)), 'orange')

#ax[1].plot(tt, cos4(tt, 10), 'r')
#plt.plot(tt, (np.cos(np.radians(70+tt))**2 * np.cos(np.radians(tt))**2)*np.max(10**(mag[:, trace, 10]/10)*2))
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)
# créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar = plt.colorbar(sm, ax=ax[0])
cbar = plt.colorbar(sm, ax=ax[1])
cbar.set_label('Indice de la fréquence')





In [ ]:
signal =10**(np.mean(mag[:, 1, :], axis=1)/10) 
signal /= np.max(signal)

popt, pcov = curve_fit(cos4, theta, signal, sigma=0.0025, absolute_sigma=True)

print("Paramètres optimisés :", popt)
error = np.sqrt(np.diag(pcov))
print("Erreurs sur les paramètres :", error)

residuals = signal - cos4(theta, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = np.sum(((signal - cos4(theta, *popt))**2) / std_res**2)

chi2_red = chi2 / (len(theta) - len(popt))
print("Chi2 réduit :", chi2_red)

plt.figure()
plt.plot(theta, signal, 'o')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 normalisé")
plt.title(r'$\theta_E$ = {:.2f} ± {:.2f}°'.format(popt[0], error[0]))
plt.grid(10)

# Loi de Malus : sans le polariseur

### Premier essai mardi 17/02 : quelques points mesurés à vue

In [ ]:
S21 = np.array([-38, -38, -38.7, -39, -39.5, -40, -41.2, -43, -46.5, -52.3, -64.8, -52.4, -47.3, -47, -44.5, -41.2, -41.7, -39])
nmeas = np.size(S21)
angle = np.arange(nmeas)*10

plt.figure()
plt.plot(angle, 10**(S21/10), 'o')

### Première automatisation mercredi 18/02 : un fichier pour chaque thetaR

In [ ]:
allthetaR = []
allmag = []

path_save2 = path_save + 'turn_thetaR_manual/'
os.chdir(path_save2)

i = 0
for file in glob.glob("*thetaR*.fits"):
    print(i)
    print(file)
    hdul = fits.open(path_save2 + file)
    header = hdul[0].header
    thetaR = header['THETA_R']
    allthetaR.append(thetaR)
    mag = hdul[0].data  # Access magnitude
    print(mag.shape)
    allmag.append(mag)
    i+=1

allthetaR = np.array(allthetaR)
allmag = np.array(allmag)
print(allmag.shape)

In [ ]:
ff = 10
colors, norm, cmap = get_colors(ff, cmap_name='viridis', vmin=0, vmax=ff-1)
trace = 0 # S21
plt.figure()
for f in range(ff):
    plt.plot(allthetaR[1:-2], 10**(allmag[1:-2, trace, f]/10), 'o', color=colors[f])

### Nouvelle optimisation : on stop le script pour tourner thetaR => 1 seul fichier

In [ ]:
allthetaR = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 
             0, -10, -20, -30, -40, -50, -60, -70, -80, -90, -100, -110, -120, -130, -140] # in degr
allthetaR2 = [-60, -30, 0, 30, 60]

hdul = fits.open(path_save + "Scan_20260220_103113_thetaR.fits")
header = hdul[0].header
hdul.info()  # View structure
mag = hdul[0].data  # Access magnitude

hdul2 = fits.open(path_save + "Scan_20260220_105809_thetaR.fits")
mag2 = hdul2[0].data  # Access magnitude

header

In [ ]:
tt = np.arange(-150, 150)
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs = axs.ravel()
axs[0].plot(allthetaR, 10**(np.mean(mag[:, 0, :], axis=1)/10), 'o-')
axs[0].plot(allthetaR2, 10**(np.mean(mag2[:, 0, :], axis=1)/10), 's-')
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')

axs[1].plot(allthetaR, 10**(np.mean(mag[:, 1, :], axis=1)/10), 'o-')
axs[1].plot(allthetaR2, 10**(np.mean(mag2[:, 1, :], axis=1)/10), 's-')
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')

axs[0].grid()
axs[1].grid()

In [ ]:
nn = 5
colors, norm, cmap = get_colors(nn, cmap_name='viridis', vmin=0, vmax=nn-1)
tt = np.arange(-120, 120)
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs = axs.ravel()
for f in [0, 10, 20, 30, 40]:
    axs[0].plot(allthetaR, 10**(mag[:, 0, f]/10), 'o', color=colors[f//10])
    axs[0].plot(allthetaR2, 10**(mag2[:, 0, f]/10), '+', color='r')
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')
axs[0].set_yscale('log')

for f in [0, 10, 20, 30, 40]:
    axs[1].plot(allthetaR, 10**(mag[:, 1, f]/10), 'o', color=colors[f//10])
    axs[1].plot(allthetaR2, 10**(mag2[:, 1, f]/10), '+', color='r')
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')
axs[1].set_yscale('log')

axs[0].grid()
axs[1].grid()

In [ ]:
  plt.figure()allthetaR = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140,

In [ ]:

for ang in [-60, -30, 0, 30, 60]:
    idx2 = np.where(np.array(allthetaR2) == ang)[0]

    if ang != 0:
        idx = np.where(np.array(allthetaR) == ang)[0]
    else:
        idx = np.where(np.array(allthetaR) == ang)[0][0]
    #print(f"Index of {ang} in allthetaR: {idx}")
    #print(f"Index of {ang} in allthetaR2: {idx2}")

    diff = np.zeros(60)
    for f in range(60):
        #diff[f] = (10**(mag[idx, 0, f]/10) - 10**(mag2[idx2, 0, f]/10))
        diff[f] = (mag[idx, 0, f] - mag2[idx2, 0, f])
    #print(f"Frequency {f}: Difference = {diff}")

    print(f"Angle {ang}°: Mean difference = {np.mean(diff):.2e}, Std difference = {np.std(diff):.2e}")




In [ ]:
plt.figure()
#plt.plot(diff, 'o')
plt.axhline(np.mean(diff), linestyle='--', label=f'Mean = {np.mean(diff):.2e}')
plt.axhline(np.mean(diff)+np.std(diff), linestyle='--')
plt.axhline(np.mean(diff)-np.std(diff), linestyle='--')
plt.xlabel('Frequency index')
plt.ylabel('Difference')
plt.title(f'Angle = {ang}° - Mean = {np.mean(diff):.2e} - Std = {np.std(diff):.2e}')
plt.legend()

# Estimation des erreurs

In [ ]:
allthetaR = []
allmag = []

#path_save2 = path_save + 'turn_thetaR_manual/'
os.chdir(path_save)

i = 0
for file in glob.glob("*20260220_14*thetaR*.fits"):
    #if i>0:
    print(i)
    print(file)
    hdul = fits.open(path_save + file)
    header = hdul[0].header
    thetaR = header['THETA_R']
    print(f"thetaR: {thetaR}")
    allthetaR.append(ast.literal_eval(thetaR))
    mag = hdul[0].data  # Access magnitude
    print(mag.shape)
    allmag.append(mag)
    i+=1

allmag = np.array(allmag)
allthetaR = np.array(allthetaR)
print(allmag.shape)

In [ ]:
freqs = [35] # indice de la frequence
nmeas = len(allthetaR)
print(nmeas)
colors, norm, cmap = get_colors(nmeas, cmap_name='plasma', vmin=0, vmax=nmeas-1)

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.ravel()
for m in range(nmeas):
    for i, f in enumerate(freqs):
        axs[0].plot(allthetaR[m], 10**(allmag[m, :, 0, f]/10), 'o', color=colors[m])
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')
#axs[0].set_yscale('log')
axs[0].grid()
for m in range(nmeas):
    for i, f in enumerate(freqs):
        axs[1].plot(allthetaR[m], 10**(allmag[m, :, 1, f]/10), 'o', color=colors[m])
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')
#axs[1].set_yscale('log')
axs[1].grid()

fig.suptitle(f'Frequency index: {freqs[0]}')



In [ ]:
mean = np.mean(allmag, axis=0)
mean.shape

std = np.std(allmag, axis=0) # [nangle, nS, nfreq]

print(std[:, 0, 0])
print(mean[:, 0, 0])

In [ ]:
plt.figure(figsize=(10, 5))
for a in range(std.shape[0]): # pour chaque angle
    plt.plot(std[a, 0, :], label=f'Std angle {a}')
plt.xlabel('Frequency index')
plt.ylabel('Standard deviation')
plt.legend()
plt.grid()

In [ ]:
allmag.shape

In [ ]:
m = 2 # indice de la mesure
m90a = allmag[m,0,0,:]
m90b = allmag[m,1,0,:]

p90a = allmag[m,5,0,:]
p90b = allmag[m,6,0,:]

plt.figure(figsize=(10, 5))
plt.plot(m90a, label='m90a')
plt.plot(m90b, label='m90b')
plt.plot(p90a, label='p90a')
plt.plot(p90b, label='p90b')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.legend()
plt.grid()

In [ ]:
diff_p90 = p90a - p90b
diff_m90 = m90a - m90b

plt.figure(figsize=(10, 5))
plt.plot(diff_p90, label='diff_p90')
plt.plot(diff_m90, label='diff_m90')
plt.xlabel('Frequency index')
plt.grid()

print(diff_p90.shape)
from astropy.stats import sigma_clipped_stats

mean_p90, median_p90, stddev_p90 = sigma_clipped_stats(diff_p90, sigma=2)
print(mean_p90, median_p90, stddev_p90)

mean_m90, median_m90, stddev_m90 = sigma_clipped_stats(diff_m90, sigma=2)
print(mean_m90, median_m90, stddev_m90)